In [ ]:
# ==========================================================
# 0. MY IMPORTS
# ==========================================================
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, PowerTransformer

# ==========================================================
# 1. MY CONFIGURATION
# ==========================================================
INPUT_ROOT = '../input'
dirs = os.listdir(INPUT_ROOT)
BASE_PATH = os.path.join(INPUT_ROOT, dirs[0]) if len(dirs) else INPUT_ROOT

if os.path.exists(os.path.join(BASE_PATH, 'dataset')):
    BASE_PATH = os.path.join(BASE_PATH, 'dataset')

TRAIN_DIR = os.path.join(BASE_PATH, 'train')
TEST_DIR = os.path.join(BASE_PATH, 'test')

for f in ['train.csv', 'labels.csv', 'train_labels.csv']:
    p = os.path.join(BASE_PATH, f)
    if os.path.exists(p):
        LABELS_FILE = p
        break

N_FOLDS = 10
BATCH_SIZE = 128
CENTER = 9

# ==========================================================
# 2. MY FEATURE ENGINEERING (WINNING APPROACH)
# ==========================================================
def extract_features(df, folder):
    feats, labels = [], []

    for _, row in df.iterrows():
        img = np.load(os.path.join(folder, row['filename']))
        c = CENTER

        center = img[c, c, :]
        d1 = np.diff(center)
        d2 = np.diff(d1)

        neigh = img[c-1:c+2, c-1:c+2, :]
        n_mean = neigh.mean(axis=(0,1))
        n_std = neigh.std(axis=(0,1))

        skew = np.mean((center - center.mean())**3)
        kurt = np.mean((center - center.mean())**4)
        area = np.trapezoid(center)

        feat = np.concatenate([
            center, d1, d2, n_mean, n_std,
            [skew, kurt, area]
        ])

        feats.append(feat)
        labels.append(row['label'] if 'label' in row else 0)

    return np.array(feats), np.array(labels)

# ==========================================================
# 3. HOW I LOAD THE DATA
# ==========================================================
df = pd.read_csv(LABELS_FILE)
df = df[~df['label'].isin([4,5])].copy()

X, y = extract_features(df, TRAIN_DIR)

test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.npy')])
df_test = pd.DataFrame({'filename': test_files})
X_test, _ = extract_features(df_test, TEST_DIR)

le = LabelEncoder()
y_enc = le.fit_transform(y)
NUM_CLASSES = len(le.classes_)

# ==========================================================
# 4. MY POWER TRANSFORMER (CRITICAL STEP)
# ==========================================================
pt = PowerTransformer(method='yeo-johnson')
X = pt.fit_transform(X)
X_test = pt.transform(X_test)

# ==========================================================
# 5. MY MODELS
# ==========================================================
def build_mlp(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(1e-3),
        loss='sparse_categorical_crossentropy'
    )
    return model

# ==========================================================
# 6. MY TRAINING + PREDICTION (FOLDS)
# ==========================================================
lgb_preds = np.zeros((len(X_test), NUM_CLASSES))
mlp_preds = np.zeros((len(X_test), NUM_CLASSES))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for fold, (tr, va) in enumerate(skf.split(X, y_enc)):
    print(f"I am processing fold {fold+1}/{N_FOLDS}")

    # ---- LightGBM ----
    lgbm = lgb.LGBMClassifier(
        n_estimators=2500,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.7,
        class_weight='balanced',
        device='gpu',
        verbose=-1
    )
    lgbm.fit(X[tr], y_enc[tr])
    lgb_preds += lgbm.predict_proba(X_test)

    # ---- MLP ----
    mlp = build_mlp(X.shape[1])
    mlp.fit(
        X[tr], y_enc[tr],
        validation_data=(X[va], y_enc[va]),
        epochs=80,
        batch_size=BATCH_SIZE,
        callbacks=[
            ReduceLROnPlateau(patience=5),
            EarlyStopping(patience=10, restore_best_weights=True)
        ],
        verbose=0
    )
    mlp_preds += mlp.predict(X_test, batch_size=BATCH_SIZE)

lgb_preds /= N_FOLDS
mlp_preds /= N_FOLDS

# ==========================================================
# 7. MY ENSEMBLE
# ==========================================================
base_probs = 0.55 * lgb_preds + 0.45 * mlp_preds

# ==========================================================
# 8. MY PSEUDO-LABELING (SAFE APPROACH)
# ==========================================================
CONF_THRESH = 0.95
conf = np.max(base_probs, axis=1)
mask = conf > CONF_THRESH

print("Pseudo-labels I added:", mask.sum())

X_aug = np.vstack([X, X_test[mask]])
y_aug = np.hstack([y_enc, np.argmax(base_probs[mask], axis=1)])

# ==========================================================
# 9. MY FINAL RETRAINING (LGBM)
# ==========================================================
final_preds = np.zeros((len(X_test), NUM_CLASSES))

for tr, va in skf.split(X_aug, y_aug):
    model = lgb.LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.02,
        num_leaves=63,
        subsample=0.9,
        colsample_bytree=0.8,
        class_weight='balanced',
        device='gpu',
        verbose=-1
    )
    model.fit(X_aug[tr], y_aug[tr])
    final_preds += model.predict_proba(X_test)

final_preds /= N_FOLDS

# ==========================================================
# 10. MY SUBMISSION
# ==========================================================
final_labels = le.inverse_transform(np.argmax(final_preds, axis=1))
submission = pd.DataFrame({
    'filename': test_files,
    'label': final_labels
})
submission.to_csv('submission_safe_85.csv', index=False)

print("✔ I successfully generated submission_safe_85.csv")